## Step 12 — create voronoi polygons for zones_3 features, intermediate step to resolve low population islands
**# of cells in notebook:** 1

**Purpose:** In Juba, and I reckon this will be true in other cities, there is a problem of small low population --or 0 population-- 'islands' on the periphery. I believe this will be a problem in other cities because the workflow that creates city extents is very generous with the perihperal areas it includes. This introduces considerable complication when merging low population features, because merging is based on a polygon neighbors table where neighbors share edges. After experientation, I believe one of the cleanest ways to resolve the issue is to create voronoi polygons across zones_3 features and then to dissolve them by the `concat` value of the features within them. Within the dissolved voronois, we then dissolve zones_3 singlepart features into multipart features and sum their populations. This notebook is strictly concerned creating the voronois. To do so, we adapt a step (morphological tesselation) from the Urban Taxonomy package to `zones_3` features.     

**Input:**

- a geodatabase with `zones_3`, `extent_hull`
  
**Output:** `zones_3_morph_tess` polygon layer

**Main logic:**

1. Read `zones_3` and `extent_hull`, check that both layers exist, have valid polygon geometry, and share a compatible CRS. If needed, `extent_hull` is reprojected to match the CRS of `zones_3`
2. Clean the input geometries by removing null/empty/non-polygon geometries, repairing invalid geometries, and exploding multipart zone geometries into singlepart features as a defensive step.
3. Create a stable `source_zone_id` for each `zones_3` feature so the tessellation output can be traced back to the original zone, and prepare a dissolved `extent_hull` geometry to use as the clipping boundary.
4. Run `momepy.morphological_tessellation` on `zones_3`, using the specified `segment` and `shrink` values, clipped to the dissolved `extent_hull`. This creates one tessellation cell associated with each source zone feature.
5. Join the original `concat` value back to each tessellation cell using `source_zone_id`, then run a validation check to confirm that each source zone’s representative point falls within a tessellation cell.
6. Clean the final tessellation geometries, calculate `area_m2`, keep only simple output fields, and write the result to `voronoi.gdb` as `zones_3_morph_tess`

In [1]:
import os
import warnings

import geopandas as gpd
import pandas as pd
import momepy
import shapely

# Optional but useful for creating the output FileGDB
try:
    import arcpy
    HAS_ARCPY = True
except Exception:
    HAS_ARCPY = False

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# Inputs
# ------------------------------------------------------------

zones_gdb = r"E:\World Bank deliverbale 1\_analysis\zones\zones.gdb"
zones_layer = "zones_3"

extent_gdb = r"E:\World Bank deliverbale 1\_analysis\roads\roads.gdb"
extent_layer = "extent_hull"

out_folder = r"E:\World Bank deliverbale 1\_analysis\voronoi"
out_gdb_name = "voronoi.gdb"
out_gdb = os.path.join(out_folder, out_gdb_name)

out_layer = "zones_3_morph_tess"

segment_value = 0.5
shrink_value = 0.4

# ------------------------------------------------------------
# Create output geodatabase if needed
# ------------------------------------------------------------

os.makedirs(out_folder, exist_ok=True)

if not os.path.exists(out_gdb):
    if HAS_ARCPY:
        print(f"Creating output geodatabase: {out_gdb}")
        arcpy.management.CreateFileGDB(out_folder, out_gdb_name)
    else:
        raise RuntimeError(
            f"Output geodatabase does not exist and arcpy is not available to create it:\n{out_gdb}"
        )
else:
    print(f"Output geodatabase already exists: {out_gdb}")

# ------------------------------------------------------------
# Read inputs
# ------------------------------------------------------------

print("Reading zones_3...")
zones = gpd.read_file(zones_gdb, layer=zones_layer)

print("Reading extent_hull...")
extent = gpd.read_file(extent_gdb, layer=extent_layer)

print(f"zones_2 feature count: {len(zones):,}")
print(f"extent_hull feature count: {len(extent):,}")

# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

if zones.empty:
    raise ValueError("zones_3 is empty.")

if extent.empty:
    raise ValueError("extent_hull is empty.")

if zones.crs is None:
    raise ValueError("zones_3 has no CRS defined.")

if extent.crs is None:
    raise ValueError("extent_hull has no CRS defined.")

print(f"zones_3 CRS: {zones.crs}")
print(f"extent_hull CRS: {extent.crs}")

# Reproject extent_hull to zones CRS if needed
if zones.crs != extent.crs:
    print("Reprojecting extent_hull to match zones_3 CRS...")
    extent = extent.to_crs(zones.crs)

# Check for concat field
if "concat" not in zones.columns:
    raise ValueError("The field 'concat' was not found in zones_3.")

# ------------------------------------------------------------
# Clean geometries
# ------------------------------------------------------------

print("Cleaning geometries...")

zones = zones.copy()
extent = extent.copy()

# Keep only polygonal geometries
zones = zones[zones.geometry.notna()].copy()
zones = zones[zones.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

extent = extent[extent.geometry.notna()].copy()
extent = extent[extent.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

# Fix invalid geometries using shapely.make_valid if available
zones["geometry"] = zones.geometry.apply(
    lambda g: shapely.make_valid(g) if not g.is_valid else g
)
extent["geometry"] = extent.geometry.apply(
    lambda g: shapely.make_valid(g) if not g.is_valid else g
)

# Explode any accidental multipart geometries in zones_2, just in case
# You said zones_2 is already singlepart, but this makes the script defensive.
zones = zones.explode(index_parts=False).reset_index(drop=True)

# Remove empty geometries
zones = zones[~zones.geometry.is_empty].copy()
extent = extent[~extent.geometry.is_empty].copy()

print(f"zones_3 feature count after cleaning/explode: {len(zones):,}")

# ------------------------------------------------------------
# Create stable source ID
# ------------------------------------------------------------

# Preserve the original row identity before tessellation.
# This is useful if you later want to join attributes back.
zones = zones.reset_index(drop=True)
zones["source_zone_id"] = zones.index.astype("int64") + 1

# Keep a lightweight source table
source_attrs = zones[["source_zone_id", "concat", "geometry"]].copy()

# ------------------------------------------------------------
# Prepare clip geometry
# ------------------------------------------------------------

print("Preparing clip geometry from extent_hull...")

# Dissolve extent_hull to a single clipping geometry
clip_geom = extent.geometry.union_all()

if clip_geom.is_empty:
    raise ValueError("The dissolved extent_hull geometry is empty.")

# momepy expects the clip argument as a polygon-like geometry
print("Clip geometry prepared.")

# ------------------------------------------------------------
# Check whether shrink would collapse any polygons
# ------------------------------------------------------------

print(f"Checking negative buffer shrink={shrink_value}...")

shrunk = zones.geometry.buffer(-shrink_value)

collapsed_count = shrunk.is_empty.sum()
multipart_after_shrink = shrunk.geom_type.isin(["MultiPolygon", "GeometryCollection"]).sum()

print(f"Polygons that would become empty after shrink: {collapsed_count:,}")
print(f"Polygons that may become multipart/collections after shrink: {multipart_after_shrink:,}")

if collapsed_count > 0:
    print(
        "\nWARNING: Some polygons collapse with shrink=0.4. "
        "The tessellation may fail or omit those features. "
        "Consider reducing shrink to 0.1 if this becomes a problem.\n"
    )

# ------------------------------------------------------------
# Run morphological tessellation
# ------------------------------------------------------------

print("Running momepy morphological tessellation...")
print(f"segment = {segment_value}")
print(f"shrink  = {shrink_value}")

# Important:
# momepy uses the input GeoDataFrame index to associate tessellation cells
# with source geometries. We set source_zone_id as the index so the output
# can be traced back clearly.
zones_for_tess = zones.set_index("source_zone_id", drop=False)

tess = momepy.morphological_tessellation(
    zones_for_tess,
    clip=clip_geom,
    segment=segment_value,
    shrink=shrink_value,
    simplify=True
)

# Depending on momepy version, output may be GeoDataFrame or GeoSeries.
if isinstance(tess, gpd.GeoSeries):
    tess = gpd.GeoDataFrame(
        {"source_zone_id": tess.index.astype("int64")},
        geometry=tess.values,
        crs=zones.crs
    )
else:
    tess = tess.copy()
    tess = tess.set_crs(zones.crs, allow_override=True)

    # If source_zone_id came through as index, restore it as a field.
    if "source_zone_id" not in tess.columns:
        tess["source_zone_id"] = tess.index.astype("int64")

tess = tess.reset_index(drop=True)

print(f"Tessellation cell count: {len(tess):,}")

# ------------------------------------------------------------
# Attach concat from source zones
# ------------------------------------------------------------

# The tessellation cells should already correspond to the source polygons.
# This join attaches concat directly from the source feature.
tess = tess.merge(
    zones[["source_zone_id", "concat"]],
    on="source_zone_id",
    how="left"
)

missing_concat = tess["concat"].isna().sum()
print(f"Tessellation cells missing concat after direct source join: {missing_concat:,}")

# ------------------------------------------------------------
# Optional validation: centroid spatial join check
# ------------------------------------------------------------

print("Running optional centroid-in-cell validation...")

zone_centroids = zones[["source_zone_id", "concat", "geometry"]].copy()

# Use representative_point instead of centroid because it is guaranteed
# to fall inside the source polygon.
zone_centroids["geometry"] = zone_centroids.geometry.representative_point()

zone_centroids = gpd.GeoDataFrame(
    zone_centroids,
    geometry="geometry",
    crs=zones.crs
)

# Spatial join: source polygon representative point within tessellation cell
check = gpd.sjoin(
    zone_centroids,
    tess[["source_zone_id", "concat", "geometry"]],
    how="left",
    predicate="within",
    lsuffix="zone",
    rsuffix="tess"
)

matched_count = check["source_zone_id_tess"].notna().sum()
print(f"Representative points matched to tessellation cells: {matched_count:,} of {len(zone_centroids):,}")

if matched_count < len(zone_centroids):
    print(
        "\nWARNING: Some source polygon representative points did not fall within a tessellation cell. "
        "This can happen if geometries collapsed during shrink or if there are clipping issues.\n"
    )

# ------------------------------------------------------------
# Final cleanup before writing
# ------------------------------------------------------------

print("Final geometry cleanup...")

tess = tess[tess.geometry.notna()].copy()
tess = tess[~tess.geometry.is_empty].copy()

tess["geometry"] = tess.geometry.apply(
    lambda g: shapely.make_valid(g) if not g.is_valid else g
)

# FileGDB has limitations with some field types.
# Convert concat to string safely.
tess["concat"] = tess["concat"].astype(str)

# Add area field for inspection
tess["area_m2"] = tess.geometry.area

# Keep output fields simple
keep_cols = ["source_zone_id", "concat", "area_m2", "geometry"]
tess_out = tess[keep_cols].copy()

print(f"Final output feature count: {len(tess_out):,}")
print(f"Total tessellation area: {tess_out.geometry.area.sum():,.2f}")

# ------------------------------------------------------------
# Write to FileGDB
# ------------------------------------------------------------

out_path_display = os.path.join(out_gdb, out_layer)

print(f"Writing output layer: {out_path_display}")

# Delete existing layer if present
if HAS_ARCPY:
    out_fc = os.path.join(out_gdb, out_layer)
    if arcpy.Exists(out_fc):
        print(f"Deleting existing layer: {out_fc}")
        arcpy.management.Delete(out_fc)

# Write using pyogrio/OpenFileGDB through GeoPandas
tess_out.to_file(
    out_gdb,
    layer=out_layer,
    driver="OpenFileGDB"
)

print("Done.")
print(f"Output written to: {out_path_display}")

Output geodatabase already exists: E:\World Bank deliverbale 1\_analysis\voronoi\voronoi.gdb
Reading zones_3...
Reading extent_hull...
zones_2 feature count: 528
extent_hull feature count: 1
zones_3 CRS: EPSG:32636
extent_hull CRS: EPSG:4326
Reprojecting extent_hull to match zones_3 CRS...
Cleaning geometries...
zones_3 feature count after cleaning/explode: 528
Preparing clip geometry from extent_hull...
Clip geometry prepared.
Checking negative buffer shrink=0.4...
Polygons that would become empty after shrink: 0
Polygons that may become multipart/collections after shrink: 0
Running momepy morphological tessellation...
segment = 0.5
shrink  = 0.4
Tessellation cell count: 528
Tessellation cells missing concat after direct source join: 0
Running optional centroid-in-cell validation...
Representative points matched to tessellation cells: 528 of 528
Final geometry cleanup...
Final output feature count: 528
Total tessellation area: 426,594,733.36
Writing output layer: E:\World Bank deliver

C:\miniconda3\envs\neatnet_env\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field source_zone_id of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
